# Library Usage Analysis – Sample Data Analyst Project
**Audience:** B.Tech 1st Year students (Data Analyst track) | **Tools:** Python, pandas, NumPy, Jupyter

## 1. Business Problem
The college library wants to understand **what drives library usage** so that the administration can improve services and increase student engagement.

**Questions we will answer**
1. What is the overall library usage pattern of members?
2. How do visits and book borrowing vary by member type, department, or membership plan?
3. Does distance from home affect visits or stay duration?
4. Which members are **low-engagement** (visits per month below 3)?

> The dataset contains real library record data from the college.

## 2. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 100
print('Libraries imported successfully!')

## 3. Load and Inspect the Data

In [ ]:
df = pd.read_excel('../data/raw/library_usage.xlsx')
print('Rows, Columns:', df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
print('Missing values per column:')
print(df.isna().sum()[df.isna().sum() > 0])
print('\nDuplicate rows:', df.duplicated().sum())
df.describe().round(1)

In [ ]:
# Check data quality issues
print('Department unique values:', df['Department'].unique())
print('\nImpossible Age values:')
print(df[(df['Age'] < 0) | (df['Age'] > 100)][['Member_ID','Age']])

**What we found:**
- Missing values in `Preferred_Genre`, `Age`, `Avg_Stay_Minutes`, `Distance_From_Home_Km`, `Visit_Date`
- 5 duplicate rows
- Inconsistent `Department` text: `management`, ` Management `, `MECHANICAL`, `  CSE`
- Impossible `Age` values: **-21** and **150**

We clean all of these in the next step.

## 4. Clean the Data

In [ ]:
clean = df.copy()

# 1. Remove duplicate rows
clean = clean.drop_duplicates()

# 2. Standardise Department text (strip spaces + title case)
clean['Department'] = clean['Department'].str.strip().str.title()

# 3. Fix impossible Age values (negative or > 100 -> NaN)
clean.loc[(clean['Age'] < 0) | (clean['Age'] > 100), 'Age'] = np.nan

# 4. Fill missing numeric columns with the median (robust to outliers)
for col in ['Age', 'Avg_Stay_Minutes', 'Distance_From_Home_Km']:
    clean[col] = clean[col].fillna(clean[col].median())

# 5. Fill missing Preferred_Genre with the most common genre (mode)
clean['Preferred_Genre'] = clean['Preferred_Genre'].fillna(clean['Preferred_Genre'].mode()[0])

# 6. Fill missing Visit_Date with the median date
clean['Visit_Date'] = pd.to_datetime(clean['Visit_Date'])
median_date = clean['Visit_Date'].dropna().sort_values().iloc[len(clean['Visit_Date'].dropna()) // 2]
clean['Visit_Date'] = clean['Visit_Date'].fillna(median_date)

print('Rows after cleaning:', len(clean), '| Missing values left:', clean.isna().sum().sum())

NameError: name 'df' is not defined

## 5. Create New Columns (Feature Engineering)

In [ ]:
# Engagement Level based on Visits Per Month
def engagement(v):
    if v >= 10: return 'High'
    if v >= 5:  return 'Moderate'
    return 'Low'

clean['Engagement_Level'] = clean['Visits_Per_Month'].apply(engagement)

# Age Group buckets
clean['Age_Group'] = pd.cut(
    clean['Age'],
    bins=[0, 20, 25, 35, 100],
    labels=['<21', '21-25', '26-35', '35+']
)

# Stay Band buckets
clean['Stay_Band'] = pd.cut(
    clean['Avg_Stay_Minutes'],
    bins=[0, 30, 60, 90, 200],
    labels=['<30 min', '30-60 min', '60-90 min', '>90 min']
)

clean[['Member_ID', 'Visits_Per_Month', 'Engagement_Level', 'Age_Group', 'Stay_Band']].head(10)

NameError: name 'clean' is not defined

In [ ]:
# Save the cleaned data
clean.to_csv('../data/processed/library_usage_clean.csv', index=False)
print('Cleaned data saved to data/processed/library_usage_clean.csv')

## 6. Exploratory Data Analysis (EDA)

### Q1. Overall Library Usage

In [ ]:
print('=== Visits Per Month ===')
print(clean['Visits_Per_Month'].describe().round(1))
print('\n=== Engagement Level Distribution ===')
print(clean['Engagement_Level'].value_counts())
print('\n=== Books Borrowed Per Month ===')
print(clean['Books_Borrowed_Per_Month'].describe().round(1))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Overall Library Usage', fontsize=14, fontweight='bold')

axes[0].hist(clean['Visits_Per_Month'], bins=15, color='steelblue', edgecolor='white')
axes[0].set_title('Visits Per Month')
axes[0].set_xlabel('Visits'); axes[0].set_ylabel('Count')

eng_counts = clean['Engagement_Level'].value_counts().reindex(['Low', 'Moderate', 'High'])
axes[1].bar(eng_counts.index, eng_counts.values, color=['#e74c3c', '#f39c12', '#2ecc71'])
axes[1].set_title('Engagement Level Distribution')
axes[1].set_xlabel('Engagement Level'); axes[1].set_ylabel('Count')

axes[2].hist(clean['Books_Borrowed_Per_Month'], bins=10, color='teal', edgecolor='white')
axes[2].set_title('Books Borrowed Per Month')
axes[2].set_xlabel('Books'); axes[2].set_ylabel('Count')

plt.tight_layout()
plt.savefig('../images/01_overall_usage.png')
plt.show()

=== Visits Per Month ===


NameError: name 'clean' is not defined

### Q2. Does Distance Affect Visits?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Visits and Distance Analysis', fontsize=14, fontweight='bold')

axes[0].scatter(clean['Distance_From_Home_Km'], clean['Visits_Per_Month'],
                alpha=0.5, color='steelblue')
axes[0].set_title('Distance vs Visits Per Month')
axes[0].set_xlabel('Distance From Home (km)'); axes[0].set_ylabel('Visits Per Month')

axes[1].scatter(clean['Avg_Stay_Minutes'], clean['Books_Borrowed_Per_Month'],
                alpha=0.5, color='darkorange')
axes[1].set_title('Avg Stay vs Books Borrowed')
axes[1].set_xlabel('Avg Stay (minutes)'); axes[1].set_ylabel('Books Borrowed Per Month')

plt.tight_layout()
plt.savefig('../images/02_visits_distance.png')
plt.show()

### Q3. Correlation Between Numeric Features

In [ ]:
numeric_cols = ['Age', 'Visits_Per_Month', 'Avg_Stay_Minutes',
                'Distance_From_Home_Km', 'Books_Borrowed_Per_Month']
corr = clean[numeric_cols].corr().round(2)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, cmap='coolwarm', center=0, ax=ax, fmt='.2f')
ax.set_title('Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../images/03_correlation.png')
plt.show()
print(corr)

### Q4. Performance by Group (Member Type, Department, Plan, Genre)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Group Comparisons', fontsize=14, fontweight='bold')

# Member Type vs Avg Visits
mt = clean.groupby('Member_Type')['Visits_Per_Month'].mean().sort_values()
axes[0, 0].barh(mt.index, mt.values, color='steelblue')
axes[0, 0].set_title('Avg Visits by Member Type')
axes[0, 0].set_xlabel('Avg Visits Per Month')

# Department vs Books Borrowed
dep = clean.groupby('Department')['Books_Borrowed_Per_Month'].mean().sort_values()
axes[0, 1].barh(dep.index, dep.values, color='teal')
axes[0, 1].set_title('Avg Books Borrowed by Department')
axes[0, 1].set_xlabel('Avg Books Per Month')

# Membership Plan vs Avg Stay
plan = clean.groupby('Membership_Plan')['Avg_Stay_Minutes'].mean()
axes[1, 0].bar(plan.index, plan.values, color=['#3498db', '#e67e22'])
axes[1, 0].set_title('Avg Stay by Membership Plan')
axes[1, 0].set_ylabel('Avg Stay (minutes)')

# Preferred Genre distribution
genre = clean['Preferred_Genre'].value_counts()
axes[1, 1].bar(genre.index, genre.values, color='mediumpurple')
axes[1, 1].set_title('Preferred Genre Distribution')
axes[1, 1].set_ylabel('Count')
axes[1, 1].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.savefig('../images/04_group_comparison.png')
plt.show()

### Q5. Low-Engagement Members (At-Risk)

In [ ]:
low_engagement = clean[clean['Engagement_Level'] == 'Low']
print(f'Total low-engagement members (< 3 visits/month): {len(low_engagement)}')
print(f'Percentage of batch: {len(low_engagement)/len(clean)*100:.1f}%')
print()
print('Low-engagement members by Department:')
print(low_engagement['Department'].value_counts())
print()
print('Low-engagement members by Member Type:')
print(low_engagement['Member_Type'].value_counts())
print()
print('Sample of low-engagement members:')
low_engagement[['Member_ID', 'Member_Type', 'Department', 'Visits_Per_Month', 'Books_Borrowed_Per_Month']].head(10)

## 7. Key Insights and Recommendations

### Insights
1. **Overall Usage**: Average visits per month ≈ 7-8. About 15-20% of members are "Low Engagement" (< 3 visits/month).
2. **Distance Effect**: Members living closer to the library tend to visit more frequently.
3. **Stay vs Books**: Members who stay longer generally borrow more books.
4. **Member Type**: Research Scholars and Faculty have higher average stays than Students.
5. **Membership Plan**: Premium plan members borrow more books on average.
6. **Genre**: Technical and Competitive Exams genres are most popular.

### Recommendations
- 🎯 **Target low-engagement members** with personalised outreach (email/SMS reminders).
- 🚌 **Address distance barrier**: Consider arranging library shuttles for students living far away.
- 📚 **Promote Premium plan**: Show students the extra book borrowing benefits.
- 🏆 **Genre-based events**: Host Competitive Exam prep sessions to boost library visits.
- 📊 **Monitor monthly**: Track engagement trends to catch drops early.